In [1]:
# Preparação para o Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    caminho_pasta = '/content/drive/MyDrive/artefatos colab'
    if os.path.exists(caminho_pasta):
        os.chdir(caminho_pasta)
        print('Diretório alterado para:', os.getcwd())
    else:
        print('ATENÇÃO: Pasta não encontrada no Drive. Verifique se o nome está correto.')
except ImportError:
    print('Não está rodando no Google Colab. Mantendo diretório atual.')


Mounted at /content/drive
Diretório alterado para: /content/drive/MyDrive/artefatos colab


# Sprint 3 - Inteligência Operacional e Relatório de Alertas
Este notebook implementa o **Sistema de Geração de Resumos Textuais de Alertas**, **Classificação Textual**, e **Relatório Operacional** conforme especificado.


In [3]:
# Instalar dependências (descomente caso necessário)
!pip install pandas numpy transformers rouge-score

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from transformers import pipeline
from rouge_score import rouge_scorer

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=8a149d57423a89489a98f7ad397c6cc1915b909bd14f9fca1d2616d5ad1279ab
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


## 1. Geração de Dados Simulados (Telemetria)

In [4]:
# Simulando leituras de temperatura e vibração de motores
np.random.seed(42)

motores = ['MT-042', 'MT-015', 'MT-088']
sensores = ['Temperatura do Enrolamento', 'Vibração do Eixo']

dados = []
agora = datetime.now()

for _ in range(20):
    motor = np.random.choice(motores)
    sensor = np.random.choice(sensores)

    if 'Temperatura' in sensor:
        baseline = 80.0
        valor = baseline + np.random.normal(0, 15)
        unidade = '°C'
    else:
        baseline = 2.5
        valor = baseline + np.random.normal(0, 1.5)
        unidade = 'mm/s'

    desvio = valor - baseline

    # Definindo severidade com base no desvio
    if 'Temperatura' in sensor:
        if desvio < 5: severidade = 'Normal'
        elif 5 <= desvio < 10: severidade = 'Leve'
        elif 10 <= desvio < 18: severidade = 'Moderado'
        else: severidade = 'Crítico'
    else:
        if desvio < 0.5: severidade = 'Normal'
        elif 0.5 <= desvio < 1.0: severidade = 'Leve'
        elif 1.0 <= desvio < 2.0: severidade = 'Moderado'
        else: severidade = 'Crítico'

    dados.append({
        'timestamp': agora - timedelta(hours=np.random.randint(1, 48)),
        'motor': motor,
        'sensor': sensor,
        'valor': round(valor, 2),
        'baseline': baseline,
        'desvio': round(desvio, 2),
        'unidade': unidade,
        'severidade': severidade
    })

df_telemetria = pd.DataFrame(dados).sort_values('timestamp').reset_index(drop=True)
# Filtrar apenas os que geraram alertas (anomalias)
df_alertas = df_telemetria[df_telemetria['severidade'] != 'Normal'].copy()
df_alertas.head()


,timestamp,motor,sensor,valor,baseline,desvio,unidade,severidade
1,2026-08-19 20:23:17.814889,MT-015,Temperatura do Enrolamento,104.63,80.0,24.63,°C,Crítico
5,2026-08-20 06:23:17.814889,MT-042,Temperatura do Enrolamento,88.73,80.0,8.73,°C,Leve
12,2026-08-21 09:23:17.814889,MT-015,Temperatura do Enrolamento,91.08,80.0,11.08,°C,Moderado
15,2026-08-21 12:23:17.814889,MT-088,Temperatura do Enrolamento,93.32,80.0,13.32,°C,Moderado
18,2026-08-21 15:23:17.814889,MT-042,Vibração do Eixo,3.09,2.5,0.59,mm/s,Leve


## 2. Geração de Resumos Textuais de Alertas (NLG por Templates)

In [5]:
def gerar_alerta(row):
    severidade = row['severidade'].lower()
    motor = row['motor']
    sensor = row['sensor']
    desvio = row['desvio']
    unidade = row['unidade']

    if severidade == 'leve':
        return f"Aviso: O motor {motor} apresenta ligeiro aumento na {sensor.lower()} (+{desvio}{unidade}). Monitoramento contínuo sugerido."
    elif severidade == 'moderado':
        return f"Alerta moderado detectado no motor {motor}. A {sensor.lower()} apresentou desvio de +{desvio}{unidade} acima do baseline. Recomenda-se verificação na próxima janela de manutenção."
    elif severidade == 'crítico':
        return f"CRÍTICO: Ação imediata requerida no motor {motor}! A {sensor.lower()} excedeu o limite seguro em +{desvio}{unidade}. Risco iminente de falha."
    else:
        return "Operação normal."

df_alertas['texto_alerta'] = df_alertas.apply(gerar_alerta, axis=1)
for t in df_alertas['texto_alerta'].head(3):
    print("-", t)


- CRÍTICO: Ação imediata requerida no motor MT-015! A temperatura do enrolamento excedeu o limite seguro em +24.63°C. Risco iminente de falha.
- Aviso: O motor MT-042 apresenta ligeiro aumento na temperatura do enrolamento (+8.73°C). Monitoramento contínuo sugerido.
- Alerta moderado detectado no motor MT-015. A temperatura do enrolamento apresentou desvio de +11.08°C acima do baseline. Recomenda-se verificação na próxima janela de manutenção.


## 3. Classificação Textual de Eventos

In [6]:
# Utilizando modelo Zero-Shot para classificar o texto do alerta
try:
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
    labels_candidatas = ["anomalia elétrica", "anomalia mecânica", "manutenção preventiva", "manutenção corretiva"]

    def classificar_evento(texto):
        # Inferindo categoria baseada no sensor e contexto (simulação rápida no zero-shot)
        res = classifier(texto, candidate_labels=labels_candidatas)
        return res['labels'][0]

    # Para demonstração, classificar os primeiros 5 alertas
    df_demo = df_alertas.head(5).copy()
    df_demo['categoria_inferida'] = df_demo['texto_alerta'].apply(classificar_evento)
    display(df_demo[['texto_alerta', 'categoria_inferida']])
except Exception as e:
    print("Erro ao carregar o classificador (verifique a internet/memória):", e)
    # Fallback simples baseado em regras
    def fallback_classifier(texto):
        if 'temperatura' in texto.lower(): return 'anomalia elétrica'
        if 'vibração' in texto.lower(): return 'anomalia mecânica'
        return 'manutenção corretiva'
    df_alertas['categoria_inferida'] = df_alertas['texto_alerta'].apply(fallback_classifier)
    display(df_alertas[['texto_alerta', 'categoria_inferida']].head(5))


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

,texto_alerta,categoria_inferida
1,CRÍTICO: Ação imediata requerida no motor MT-0...,manutenção preventiva
5,Aviso: O motor MT-042 apresenta ligeiro aument...,manutenção preventiva
12,Alerta moderado detectado no motor MT-015. A t...,manutenção preventiva
15,Alerta moderado detectado no motor MT-088. A t...,manutenção preventiva
18,Aviso: O motor MT-042 apresenta ligeiro aument...,manutenção preventiva


## 4. Relatório de Estado Operacional

In [7]:
def gerar_relatorio(df):
    total_alertas = len(df)
    criticos = len(df[df['severidade'] == 'Crítico'])
    motores_afetados = df['motor'].unique()

    relatorio = f"**Relatório Operacional Diário**\n"
    relatorio += f"Nas últimas 24 horas, foram registrados {total_alertas} alertas de anomalia.\n"
    relatorio += f"Identificamos {criticos} eventos de severidade CRÍTICA.\n\n"
    relatorio += "Equipamentos afetados:\n"
    for motor in motores_afetados:
        alertas_motor = df[df['motor'] == motor]
        relatorio += f"- **{motor}**: {len(alertas_motor)} alerta(s).\n"
        for _, row in alertas_motor.iterrows():
            relatorio += f"  - {row['sensor']}: desvio de +{row['desvio']}{row['unidade']} ({row['severidade']}).\n"

    relatorio += "\nRecomendações preliminares: Agendar inspeção imediata para equipamentos com alertas críticos e revisão do sistema de resfriamento/lubrificação."
    return relatorio

texto_relatorio = gerar_relatorio(df_alertas)
print(texto_relatorio.replace('\n', '\n'))


**Relatório Operacional Diário**
Nas últimas 24 horas, foram registrados 6 alertas de anomalia.
Identificamos 1 eventos de severidade CRÍTICA.

Equipamentos afetados:
- **MT-015**: 2 alerta(s).
  - Temperatura do Enrolamento: desvio de +24.63°C (Crítico).
  - Temperatura do Enrolamento: desvio de +11.08°C (Moderado).
- **MT-042**: 3 alerta(s).
  - Temperatura do Enrolamento: desvio de +8.73°C (Leve).
  - Vibração do Eixo: desvio de +0.59mm/s (Leve).
  - Vibração do Eixo: desvio de +1.23mm/s (Moderado).
- **MT-088**: 1 alerta(s).
  - Temperatura do Enrolamento: desvio de +13.32°C (Moderado).

Recomendações preliminares: Agendar inspeção imediata para equipamentos com alertas críticos e revisão do sistema de resfriamento/lubrificação.


## 5. Avaliação do Texto Gerado (Métricas ROUGE)

In [8]:
ground_truth = (
    "Relatório Operacional Diário\n"
    "Nas últimas 24 horas, foram registrados vários alertas de anomalia.\n"
    "Identificamos eventos de severidade crítica requerendo atenção.\n\n"
    "Equipamentos afetados apresentaram desvios significativos em sensores de temperatura e vibração.\n"
    "Recomenda-se agendar inspeção imediata para equipamentos com alertas críticos e revisão dos sistemas."
)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(ground_truth, texto_relatorio)

print("ROUGE Scores (Relatório vs Ground Truth):")
for key, value in scores.items():
    print(f"{key.upper()}: Precision={value.precision:.2f}, Recall={value.recall:.2f}, Fmeasure={value.fmeasure:.2f}")


ROUGE Scores (Relatório vs Ground Truth):
ROUGE1: Precision=0.36, Recall=0.80, Fmeasure=0.50
ROUGE2: Precision=0.26, Recall=0.58, Fmeasure=0.36
ROUGEL: Precision=0.36, Recall=0.78, Fmeasure=0.49
